# Phase 1.2: FULL detector runs, D1 / D3 / D4 (Colab, **GPU runtime: T4**, ~2 h)
Preregistration frozen at `2e92b8b`. Runs every allowed detector × dataset on the full data:
UTKFace (23,705 records) and FairFace-val (10,954). D3 runs on UTKFace only. D5 comes in a separate notebook.

**Resumable:** each finished run is copied to Drive at once; re-running skips runs already on Drive.
If Colab disconnects, reconnect (T4) and **Run all** again.

**Ethics (EXPERIMENT_PLAN §7):** never display images. Per-image outputs stay on Drive `per_image/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, shutil, subprocess
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/child-detection-audit')
REPO = pathlib.Path('/content/child-detection-skin-tone-audit')
os.environ['CHILD_AUDIT_ENV'] = 'colab'
os.environ.pop('UTKFACE_DIR', None); os.environ.pop('FAIRFACE_DIR', None)  # full data, not dev
os.chdir('/content')
!rm -rf {REPO} && git clone -q https://github.com/isaacmutuma/child-skintone-detection.git {REPO}
!git -C {REPO} log --oneline -1
os.chdir(REPO)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU: switch runtime to T4'
RUNS_DRIVE = DRIVE_ROOT / 'results' / 'phase1' / 'runs'; RUNS_DRIVE.mkdir(parents=True, exist_ok=True)
PER_IMAGE = DRIVE_ROOT / 'per_image'
EXPECTED_N = {'utkface': 23705, 'fairface_val': 10954}

def done(det, ds):
    return (RUNS_DRIVE / f'{det}__{ds}__full.json').exists() and \
           (PER_IMAGE / f'per_image__{det}__{ds}__full.csv').exists()

def save(det, ds):
    src = REPO / 'results' / 'phase1' / 'runs' / f'{det}__{ds}__full.json'
    d = json.loads(src.read_text())
    assert d['n_images'] == EXPECTED_N[ds], f"{det}/{ds}: n_images {d['n_images']} != {EXPECTED_N[ds]}"
    assert d['git']['dirty'] is False, 'code was modified in this runtime'
    shutil.copy(src, RUNS_DRIVE / src.name)
    print(f"saved {src.name}: n={d['n_images']} flagged={d['n_flagged_child']} "
          f"fails={d['fail_reasons']} {d['seconds']}s commit={d['git']['commit'][:7]}")

## 1. Unpack data from Drive `raw/` → `/content/data` (SHA256-verified, same layout as Phase 0)

In [ ]:
!python scripts/unpack_data.py

## 2. Main environment (D3, D4)

In [ ]:
!pip -q install open_clip_torch huggingface_hub
try:
    import dlib; print('dlib', dlib.__version__, 'cuda:', dlib.DLIB_USE_CUDA)
except ImportError:
    print('dlib missing -> installing (slow)'); subprocess.run(['pip', '-q', 'install', 'dlib'], check=True)
W = DRIVE_ROOT / 'weights' / 'fairface'
for f in ['mmod_human_face_detector.dat', 'shape_predictor_5_face_landmarks.dat']:
    if not (W / f).exists():
        W.mkdir(parents=True, exist_ok=True)
        !wget -q http://dlib.net/files/{f}.bz2 -O /content/{f}.bz2 && bunzip2 -f /content/{f}.bz2 && cp /content/{f} {W}/
!ls -la {W}

## 3. D4 (CLIP B/32, L/14) on both datasets, D3 on UTKFace

In [ ]:
runs = [('clip_vitb32', 'utkface'), ('clip_vitb32', 'fairface_val'),
        ('clip_vitl14', 'utkface'), ('clip_vitl14', 'fairface_val'),
        ('fairface_clf', 'utkface')]
for det, ds in runs:
    if done(det, ds):
        print(f'skip {det}/{ds} (already on Drive)'); continue
    print(f'=== {det} / {ds}')
    !python scripts/run_detector.py --detector {det} --dataset {ds} --batch-size 64
    save(det, ds)

## 4. D1 (MiVOLO v2) in its isolated venv

In [ ]:
PY = '/content/venv_mivolo/bin/python'
if not all(done('mivolo_face', ds) for ds in EXPECTED_N):
    !pip -q install uv
    !rm -rf /content/venv_mivolo && uv venv -q --python 3.11 /content/venv_mivolo
    !uv pip install -q --python {PY} --index-strategy unsafe-best-match \
        --extra-index-url https://download.pytorch.org/whl/cu121 \
        torch==2.5.1 torchvision==0.20.1 "numpy<2" "timm==0.8.13.dev0" "transformers==4.51.0" \
        "accelerate==1.8.1" "ultralytics==8.1.0" "lapx>=0.5.2" huggingface_hub pandas scipy pillow \
        opencv-python-headless omegaconf "setuptools<70" wheel
    !uv pip install -q --python {PY} --no-build-isolation --no-deps \
        "git+https://github.com/WildChlamydia/MiVOLO.git@37475e3f8818b5f22448003feec3e64b01bfb188"
    !MPLBACKEND=Agg {PY} -c "import torch, mivolo, ultralytics, omegaconf; print('mivolo env ok', torch.__version__, torch.cuda.is_available())"

In [ ]:
for ds in ['utkface', 'fairface_val']:
    if done('mivolo_face', ds):
        print(f'skip mivolo_face/{ds} (already on Drive)'); continue
    print(f'=== mivolo_face / {ds}')
    !MPLBACKEND=Agg {PY} scripts/run_detector.py --detector mivolo_face --dataset {ds} --batch-size 32
    save('mivolo_face', ds)

## 5. Check: all 7 full runs on Drive

In [ ]:
expected = [('clip_vitb32','utkface'), ('clip_vitb32','fairface_val'), ('clip_vitl14','utkface'),
            ('clip_vitl14','fairface_val'), ('fairface_clf','utkface'),
            ('mivolo_face','utkface'), ('mivolo_face','fairface_val')]
missing = [r for r in expected if not done(*r)]
print('ALL 7 FULL RUNS DONE' if not missing else f'MISSING: {missing} -> Run all again')
!cp results/env.json {DRIVE_ROOT}/results/env_colab_full.json